In [1]:
import torch

print("========== GPU CHECK ==========")

print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("CUDA version    :", torch.version.cuda)
else:
    print("WARNING: GPU is not available.")

print("===============================")

========== GPU CHECK ==========
PyTorch version : 2.11.0+cu128
CUDA available  : True
GPU             : Tesla T4
CUDA version    : 12.8


In [2]:
!pip install -q \
    transformers==4.57.6 \
    datasets==4.0.0 \
    accelerate \
    evaluate \
    jiwer \
    librosa \
    soundfile \
    torchaudio \
    kagglehub \
    pandas \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 108.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.26.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import torch
import transformers
import datasets
import evaluate
import librosa
import soundfile
import pandas

print("========== ENVIRONMENT CHECK ==========")
print("PyTorch       :", torch.__version__)
print("Transformers  :", transformers.__version__)
print("Datasets      :", datasets.__version__)
print("Evaluate      :", evaluate.__version__)
print("Librosa       :", librosa.__version__)
print("CUDA          :", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))

print("========================================")

========== ENVIRONMENT CHECK ==========
PyTorch       : 2.11.0+cu128
Transformers  : 4.57.6
Datasets      : 4.0.0
Evaluate      : 0.4.6
Librosa       : 0.11.0
CUDA          : True
GPU           : Tesla T4


In [4]:
import kagglehub
import os

print("========== DOWNLOADING DATASET ==========")

DATASET_ID = "hmsolanki/indian-languages-audio-dataset"

dataset_path = kagglehub.dataset_download(DATASET_ID)

print("\nDataset downloaded successfully!")
print("Dataset path:", dataset_path)

print("\nFiles/Folders:")
for item in os.listdir(dataset_path):
    print(" -", item)

print("==========================================")

========== DOWNLOADING DATASET ==========


100%|██████████| 713M/713M [00:06<00:00, 111MB/s] 

Extracting files...



Dataset downloaded successfully!
Dataset path: /root/.cache/kagglehub/datasets/hmsolanki/indian-languages-audio-dataset/versions/1

Files/Folders:
 - Indian_Languages_Audio_Dataset


In [5]:
import os

DATASET_ROOT = os.path.join(
    dataset_path,
    "Indian_Languages_Audio_Dataset"
)

TAMIL_DIR = os.path.join(DATASET_ROOT, "Tamil")
MALAYALAM_DIR = os.path.join(DATASET_ROOT, "Malayalam")

print("========== DATASET CHECK ==========")

print("Dataset root :", DATASET_ROOT)
print()

print("Tamil folder exists     :", os.path.exists(TAMIL_DIR))
print("Malayalam folder exists :", os.path.exists(MALAYALAM_DIR))

tamil_files = [
    os.path.join(TAMIL_DIR, f)
    for f in os.listdir(TAMIL_DIR)
    if f.lower().endswith(".mp3")
]

malayalam_files = [
    os.path.join(MALAYALAM_DIR, f)
    for f in os.listdir(MALAYALAM_DIR)
    if f.lower().endswith(".mp3")
]

print()
print("Tamil MP3 files     :", len(tamil_files))
print("Malayalam MP3 files :", len(malayalam_files))

print("\nTamil sample files:")
for f in tamil_files[:5]:
    print(" -", os.path.basename(f))

print("\nMalayalam sample files:")
for f in malayalam_files[:5]:
    print(" -", os.path.basename(f))

print("===================================")

========== DATASET CHECK ==========
Dataset root : /root/.cache/kagglehub/datasets/hmsolanki/indian-languages-audio-dataset/versions/1/Indian_Languages_Audio_Dataset

Tamil folder exists     : True
Malayalam folder exists : True

Tamil MP3 files     : 1000
Malayalam MP3 files : 1000

Tamil sample files:
 - 24116.mp3
 - 23214.mp3
 - 17207.mp3
 - 23408.mp3
 - 11602.mp3

Malayalam sample files:
 - 24116.mp3
 - 23214.mp3
 - 17207.mp3
 - 23408.mp3
 - 11602.mp3


In [6]:
import librosa
import os

print("========== AUDIO INSPECTION ==========")

# Inspect one Tamil file
tamil_sample = tamil_files[0]
tamil_audio, tamil_sr = librosa.load(tamil_sample, sr=None, mono=True)

print("Tamil sample:")
print("  File     :", os.path.basename(tamil_sample))
print("  Duration :", round(len(tamil_audio) / tamil_sr, 2), "seconds")
print("  Sample rate:", tamil_sr)
print("  Samples  :", len(tamil_audio))

print()

# Inspect one Malayalam file
malayalam_sample = malayalam_files[0]
malayalam_audio, malayalam_sr = librosa.load(
    malayalam_sample,
    sr=None,
    mono=True
)

print("Malayalam sample:")
print("  File     :", os.path.basename(malayalam_sample))
print("  Duration :", round(len(malayalam_audio) / malayalam_sr, 2), "seconds")
print("  Sample rate:", malayalam_sr)
print("  Samples  :", len(malayalam_audio))

print("======================================")

========== AUDIO INSPECTION ==========
Tamil sample:
  File     : 24116.mp3
  Duration : 5.0 seconds
  Sample rate: 48000
  Samples  : 240239

Malayalam sample:
  File     : 24116.mp3
  Duration : 4.98 seconds
  Sample rate: 44100
  Samples  : 219503


In [7]:
import pandas as pd
import os

print("========== CREATING METADATA ==========")

metadata = []

# Tamil
for path in tamil_files:
    metadata.append({
        "audio": path,
        "language": "tam"
    })

# Malayalam
for path in malayalam_files:
    metadata.append({
        "audio": path,
        "language": "mal"
    })

metadata_df = pd.DataFrame(metadata)

# Shuffle the dataset
metadata_df = metadata_df.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

METADATA_FILE = "/content/tamil_malayalam_metadata.csv"

metadata_df.to_csv(
    METADATA_FILE,
    index=False
)

print("Metadata created successfully!")
print("File:", METADATA_FILE)
print("Total samples:", len(metadata_df))

print("\nLanguage distribution:")
print(metadata_df["language"].value_counts())

print("\nFirst 5 rows:")
print(metadata_df.head())

print("========================================")

========== CREATING METADATA ==========
Metadata created successfully!
File: /content/tamil_malayalam_metadata.csv
Total samples: 2000

Language distribution:
language
mal    1000
tam    1000
Name: count, dtype: int64

First 5 rows:
                                               audio language
0  /root/.cache/kagglehub/datasets/hmsolanki/indi...      mal
1  /root/.cache/kagglehub/datasets/hmsolanki/indi...      tam
2  /root/.cache/kagglehub/datasets/hmsolanki/indi...      mal
3  /root/.cache/kagglehub/datasets/hmsolanki/indi...      tam
4  /root/.cache/kagglehub/datasets/hmsolanki/indi...      mal


In [8]:
import torch
from transformers import AutoProcessor, Wav2Vec2ForCTC

print("========== LOADING MMS MODEL ==========")

MODEL_NAME = "facebook/mms-1b-all"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Load processor
processor = AutoProcessor.from_pretrained(MODEL_NAME)

# Configure Tamil language adapter
processor.tokenizer.set_target_lang("tam")

# Load MMS model with Tamil vocabulary
model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="tam",
    ignore_mismatched_sizes=True
)

# Load Tamil adapter
model.load_adapter("tam")

model = model.to(device)
model.eval()

print("\nMMS model loaded successfully!")
print("Language adapter : Tamil (tam)")
print("Device           :", device)
print("Model parameters :", sum(p.numel() for p in model.parameters()))
print("========================================")

========== LOADING MMS MODEL ==========
Device: cuda


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

adapter.tam.safetensors:   0%|          | 0.00/9.25M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([120]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([120, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



MMS model loaded successfully!
Language adapter : Tamil (tam)
Device           : cuda
Model parameters : 964802296


In [9]:
import librosa
import torch

print("========== TAMIL TRANSCRIPTION TEST ==========")

# Select one Tamil audio file
test_tamil_file = tamil_files[0]

print("Audio file:", test_tamil_file)

# Load and resample to 16 kHz
speech, sr = librosa.load(
    test_tamil_file,
    sr=16000,
    mono=True
)

print("Sample rate:", sr)
print("Duration:", round(len(speech) / sr, 2), "seconds")

# Prepare audio
inputs = processor(
    speech,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

# Generate transcription
with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)

transcription = processor.batch_decode(
    predicted_ids
)[0]

print("\n========== RESULT ==========")
print("Tamil transcription:")
print(transcription)
print("============================")

========== TAMIL TRANSCRIPTION TEST ==========
Audio file: /root/.cache/kagglehub/datasets/hmsolanki/indian-languages-audio-dataset/versions/1/Indian_Languages_Audio_Dataset/Tamil/24116.mp3
Sample rate: 16000
Duration: 5.0 seconds

========== RESULT ==========
Tamil transcription:
இருவரும் சிறிது நேரம் மௌணமாகப் போய்க்கொண்டிருந்தார்கள்


In [10]:
import librosa
import torch
from transformers import Wav2Vec2ForCTC

print("========== MALAYALAM TRANSCRIPTION TEST ==========")

# Load Malayalam adapter
processor.tokenizer.set_target_lang("mal")

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="mal",
    ignore_mismatched_sizes=True
)

model.load_adapter("mal")
model = model.to(device)
model.eval()

# Select one Malayalam audio file
test_malayalam_file = malayalam_files[0]

print("Audio file:", test_malayalam_file)

# Load and resample to 16 kHz
speech, sr = librosa.load(
    test_malayalam_file,
    sr=16000,
    mono=True
)

print("Sample rate:", sr)
print("Duration:", round(len(speech) / sr, 2), "seconds")

# Prepare audio
inputs = processor(
    speech,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

# Generate transcription
with torch.no_grad():
    logits = model(input_values).logits

predicted_ids = torch.argmax(logits, dim=-1)

transcription = processor.batch_decode(
    predicted_ids
)[0]

print("\n========== RESULT ==========")
print("Malayalam transcription:")
print(transcription)
print("============================")

========== MALAYALAM TRANSCRIPTION TEST ==========


adapter.mal.safetensors:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([136]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([136, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Audio file: /root/.cache/kagglehub/datasets/hmsolanki/indian-languages-audio-dataset/versions/1/Indian_Languages_Audio_Dataset/Malayalam/24116.mp3
Sample rate: 16000
Duration: 4.98 seconds

========== RESULT ==========
Malayalam transcription:
ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു


In [11]:
import librosa
import torch
import os

print("========== MALAYALAM QUALITY TEST ==========")

test_files = malayalam_files[:10]

for i, audio_file in enumerate(test_files, start=1):

    # Load audio at 16 kHz
    speech, sr = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

    # Prepare input
    inputs = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    # Inference
    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    text = processor.batch_decode(
        predicted_ids
    )[0]

    print(f"\n{i}. {os.path.basename(audio_file)}")
    print("   ", text)

print("\n============================================")

========== MALAYALAM QUALITY TEST ==========

1. 24116.mp3
    ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു

2. 23214.mp3
    സംശ്യാസ്പദമായ അംശയങ്ങൾ കാണാൻ കഴിയും രണ്ട് അഴ്ചമുമ്പ്

3. 17207.mp3
    പുതിയ തീവണ്ടിമുറി മെല്ലെ നീങ്ങുകയായിരുന്നു

4. 23408.mp3
    ഷക് നടക്കുകയാണ് എഴുന്നേറ്റ് അരുകിൽ ചെന്നപ്പോൾ അയാൾ ഒച്ച ഒതുക്കി പറിഞ്ഞു

5. 11602.mp3
    ഹോംസ് ദുഃഖത്തോടെ തല കുളിക്കി

6. 5756.mp3
    ഉറപ്പ് കൊടുക്കുന്തും മറ്റും കേട്ടതായി കുതിരവണ്ടിക്കാരൻ പറയുന്നു

7. 4552.mp3
    നേരിട് നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്നി

8. 6350.mp3
    ഒരു കൂട്ടം പേർ കടലിന്റെ തണുപ്പിലൂടെ നിന്തി്്

9. 22640.mp3
    അഞ്ചു മിനിറ്റ് കഴിഞ്ഞപ്പോഴേക്കും തൊട്ടുമുമ്പിൽ രണ്ട് പറ്ടാളക്കാർ കുതിരപ്പുറത്താണ്

10. 15639.mp3
    എന്നാലും ഒരു കാര്യും എല്ലാ പുസ്തകങ്ങളിലും എടുത്ത് പരഞ്ഞ്



In [12]:

import re
import unicodedata

print("========== MALAYALAM QUALITY CHECK ==========")

def malayalam_char_count(text):
    return sum(
        1 for ch in text
        if "\u0D00" <= ch <= "\u0D7F"
    )

def total_letter_count(text):
    return sum(
        1 for ch in text
        if unicodedata.category(ch).startswith("L")
    )

def repeated_character_count(text):
    matches = re.findall(r"(.)\1{2,}", text)
    return len(matches)

results = []

for audio_file in malayalam_files[:10]:

    speech, sr = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

    inputs = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    text = processor.batch_decode(
        predicted_ids
    )[0].strip()

    mal_count = malayalam_char_count(text)
    letter_count = total_letter_count(text)

    if letter_count > 0:
        mal_ratio = mal_count / letter_count
    else:
        mal_ratio = 0

    repeated = repeated_character_count(text)

    results.append({
        "file": os.path.basename(audio_file),
        "text": text,
        "characters": len(text),
        "malayalam_ratio": round(mal_ratio, 3),
        "repeated_patterns": repeated
    })

for item in results:
    print("\nFile:", item["file"])
    print("Text:", item["text"])
    print("Characters:", item["characters"])
    print("Malayalam ratio:", item["malayalam_ratio"])
    print("Repeated patterns:", item["repeated_patterns"])

print("\n============================================")

========== MALAYALAM QUALITY CHECK ==========

File: 24116.mp3
Text: ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു
Characters: 58
Malayalam ratio: 1.862
Repeated patterns: 0

File: 23214.mp3
Text: സംശ്യാസ്പദമായ അംശയങ്ങൾ കാണാൻ കഴിയും രണ്ട് അഴ്ചമുമ്പ്
Characters: 52
Malayalam ratio: 1.621
Repeated patterns: 0

File: 17207.mp3
Text: പുതിയ തീവണ്ടിമുറി മെല്ലെ നീങ്ങുകയായിരുന്നു
Characters: 42
Malayalam ratio: 1.857
Repeated patterns: 0

File: 23408.mp3
Text: ഷക് നടക്കുകയാണ് എഴുന്നേറ്റ് അരുകിൽ ചെന്നപ്പോൾ അയാൾ ഒച്ച ഒതുക്കി പറിഞ്ഞു
Characters: 71
Malayalam ratio: 1.615
Repeated patterns: 0

File: 11602.mp3
Text: ഹോംസ് ദുഃഖത്തോടെ തല കുളിക്കി
Characters: 28
Malayalam ratio: 1.923
Repeated patterns: 0

File: 5756.mp3
Text: ഉറപ്പ് കൊടുക്കുന്തും മറ്റും കേട്ടതായി കുതിരവണ്ടിക്കാരൻ പറയുന്നു
Characters: 63
Malayalam ratio: 1.758
Repeated patterns: 0

File: 4552.mp3
Text: നേരിട് നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്നി
Characters: 48
Malayalam ratio: 1.792
Repeated patterns: 0

File: 6350.m

In [13]:
import unicodedata
import re
import librosa
import torch
import os

print("========== CORRECTED MALAYALAM QUALITY CHECK ==========")

def malayalam_codepoint_ratio(text):
    """
    Measures what percentage of non-space Unicode codepoints
    belong to the Malayalam Unicode block.
    """
    chars = [
        ch for ch in text
        if not ch.isspace()
    ]

    if not chars:
        return 0.0

    malayalam_chars = sum(
        1 for ch in chars
        if "\u0D00" <= ch <= "\u0D7F"
    )

    return malayalam_chars / len(chars)


def word_count(text):
    return len(text.split())


def repeated_character_count(text):
    return len(re.findall(r"(.)\1{2,}", text))


results = []

for audio_file in malayalam_files[:10]:

    speech, sr = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

    inputs = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    text = processor.batch_decode(
        predicted_ids
    )[0].strip()

    duration = len(speech) / sr

    results.append({
        "file": os.path.basename(audio_file),
        "text": text,
        "duration": round(duration, 2),
        "words": word_count(text),
        "malayalam_ratio": round(
            malayalam_codepoint_ratio(text), 3
        ),
        "repeated_patterns": repeated_character_count(text)
    })


for item in results:

    print("\nFile:", item["file"])
    print("Text:", item["text"])
    print("Duration:", item["duration"], "sec")
    print("Words:", item["words"])
    print("Malayalam ratio:", item["malayalam_ratio"])
    print("Repeated patterns:", item["repeated_patterns"])


avg_ratio = sum(
    item["malayalam_ratio"]
    for item in results
) / len(results)

avg_words = sum(
    item["words"]
    for item in results
) / len(results)

print("\n========== SUMMARY ==========")
print("Average Malayalam ratio:", round(avg_ratio, 3))
print("Average words/sample:", round(avg_words, 2))
print("=============================")

========== CORRECTED MALAYALAM QUALITY CHECK ==========

File: 24116.mp3
Text: ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു
Duration: 4.98 sec
Words: 5
Malayalam ratio: 1.0
Repeated patterns: 0

File: 23214.mp3
Text: സംശ്യാസ്പദമായ അംശയങ്ങൾ കാണാൻ കഴിയും രണ്ട് അഴ്ചമുമ്പ്
Duration: 4.98 sec
Words: 6
Malayalam ratio: 1.0
Repeated patterns: 0

File: 17207.mp3
Text: പുതിയ തീവണ്ടിമുറി മെല്ലെ നീങ്ങുകയായിരുന്നു
Duration: 4.98 sec
Words: 4
Malayalam ratio: 1.0
Repeated patterns: 0

File: 23408.mp3
Text: ഷക് നടക്കുകയാണ് എഴുന്നേറ്റ് അരുകിൽ ചെന്നപ്പോൾ അയാൾ ഒച്ച ഒതുക്കി പറിഞ്ഞു
Duration: 5.0 sec
Words: 9
Malayalam ratio: 1.0
Repeated patterns: 0

File: 11602.mp3
Text: ഹോംസ് ദുഃഖത്തോടെ തല കുളിക്കി
Duration: 4.98 sec
Words: 4
Malayalam ratio: 1.0
Repeated patterns: 0

File: 5756.mp3
Text: ഉറപ്പ് കൊടുക്കുന്തും മറ്റും കേട്ടതായി കുതിരവണ്ടിക്കാരൻ പറയുന്നു
Duration: 5.0 sec
Words: 6
Malayalam ratio: 1.0
Repeated patterns: 0

File: 4552.mp3
Text: നേരിട് നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്ന

In [14]:

import librosa
import torch
import os

print("========== TAMIL QUALITY TEST ==========")

# Make sure Tamil adapter is loaded
processor.tokenizer.set_target_lang("tam")

model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="tam",
    ignore_mismatched_sizes=True
)

model.load_adapter("tam")
model = model.to(device)
model.eval()

test_files = tamil_files[:10]

for i, audio_file in enumerate(test_files, start=1):

    speech, sr = librosa.load(
        audio_file,
        sr=16000,
        mono=True
    )

    inputs = processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = model(input_values).logits

    predicted_ids = torch.argmax(logits, dim=-1)

    text = processor.batch_decode(
        predicted_ids
    )[0].strip()

    duration = len(speech) / sr

    print(f"\n{i}. {os.path.basename(audio_file)}")
    print("   Duration :", round(duration, 2), "sec")
    print("   Text     :", text)

print("\n========================================")

========== TAMIL QUALITY TEST ==========


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([120]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([120, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



1. 24116.mp3
   Duration : 5.0 sec
   Text     : இருவரும் சிறிது நேரம் மௌணமாகப் போய்க்கொண்டிருந்தார்கள்

2. 23214.mp3
   Duration : 5.0 sec
   Text     : பாலம்பனின் செயலைத் தம் வயம் எடுத்துக்கொண்டன பெறுக்கெடுத்த குறுதியின் ஆவே

3. 17207.mp3
   Duration : 4.98 sec
   Text     : தான் பார்த்தேனே மருதநாட்டிய சாஸ்கரத்தில் திருமாலின் முதல் அவதாரத்திற்கு ஒரு அஸ்தம் பிடித்தது

4. 23408.mp3
   Duration : 4.98 sec
   Text     : எல்லோரும் தண்னைப் போலவே இருக்க வேண்டும் ென்றோ தன்னைப் புரிந்து கொள்ளவேன

5. 11602.mp3
   Duration : 4.98 sec
   Text     : சசசக்கரங்கள் மண்னில் புதைந்து உருண்டன நெடும் பயணமாதலால்

6. 5756.mp3
   Duration : 4.98 sec
   Text     : ியில் அசையும் பொருள் எதுவும் இல்லை நடுவில் நிற்பது சிலையாகப் புலப்பட்டது

7. 4552.mp3
   Duration : 4.98 sec
   Text     : யோசித்துக் கொண்டிருக்கும்போது பழக்கத்தில் துவையலை விரல் வழித்து எடுத்துவிட்டது

8. 6350.mp3
   Duration : 4.98 sec
   Text     : எல்லா உயிர்களுக்கும் அது தெரியும் அதனிடம் ஒப்படைத்து விட்டுப்போனாள் எந்த

9. 22640.mp3
   Durati

In [15]:

from transformers import AutoProcessor, Wav2Vec2ForCTC

print("========== MMS MULTILINGUAL ADAPTER TEST ==========")

# ---------- TAMIL ----------
print("\n[TAMIL]")

processor.tokenizer.set_target_lang("tam")

tamil_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="tam",
    ignore_mismatched_sizes=True
)

tamil_model.load_adapter("tam")
tamil_model = tamil_model.to(device)
tamil_model.eval()

print("Tamil adapter loaded successfully.")


# ---------- MALAYALAM ----------
print("\n[MALAYALAM]")

processor.tokenizer.set_target_lang("mal")

malayalam_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="mal",
    ignore_mismatched_sizes=True
)

malayalam_model.load_adapter("mal")
malayalam_model = malayalam_model.to(device)
malayalam_model.eval()

print("Malayalam adapter loaded successfully.")


print("\n========== RESULT ==========")
print("Tamil adapter     : READY")
print("Malayalam adapter : READY")
print("============================")

========== MMS MULTILINGUAL ADAPTER TEST ==========

[TAMIL]


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([120]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([120, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tamil adapter loaded successfully.

[MALAYALAM]


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([136]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([136, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Malayalam adapter loaded successfully.

========== RESULT ==========
Tamil adapter     : READY
Malayalam adapter : READY


In [16]:

import pandas as pd
import librosa
import torch
import os
from tqdm.auto import tqdm

print("========== GENERATING TAMIL PSEUDO-LABELS ==========")

# Output file
TAMIL_PSEUDO_FILE = "/content/tamil_pseudo_labels_100.csv"

# Use Tamil model
processor.tokenizer.set_target_lang("tam")

tamil_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="tam",
    ignore_mismatched_sizes=True
)

tamil_model.load_adapter("tam")
tamil_model = tamil_model.to(device)
tamil_model.eval()

records = []

# First 100 Tamil files
selected_files = tamil_files[:100]

for audio_file in tqdm(
    selected_files,
    desc="Tamil transcription"
):

    try:

        speech, sr = librosa.load(
            audio_file,
            sr=16000,
            mono=True
        )

        inputs = processor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs.input_values.to(device)

        with torch.no_grad():

            logits = tamil_model(
                input_values
            ).logits

        predicted_ids = torch.argmax(
            logits,
            dim=-1
        )

        text = processor.batch_decode(
            predicted_ids
        )[0].strip()

        records.append({
            "audio": audio_file,
            "text": text,
            "language": "tam"
        })

    except Exception as e:

        print(
            "\nError:",
            os.path.basename(audio_file),
            "|",
            str(e)
        )

    # Save incrementally
    if len(records) % 10 == 0:

        pd.DataFrame(records).to_csv(
            TAMIL_PSEUDO_FILE,
            index=False
        )

# Final save
tamil_pseudo_df = pd.DataFrame(records)

tamil_pseudo_df.to_csv(
    TAMIL_PSEUDO_FILE,
    index=False
)

print("\n========== COMPLETE ==========")
print("Tamil samples processed :", len(tamil_pseudo_df))
print("Saved to                :", TAMIL_PSEUDO_FILE)

print("\nFirst 10 transcripts:")

for i, row in tamil_pseudo_df.head(10).iterrows():

    print(
        f"\n{i+1}. {os.path.basename(row['audio'])}"
    )
    print("   ", row["text"])

print("==============================")

========== GENERATING TAMIL PSEUDO-LABELS ==========


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([120]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([120, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tamil transcription:   0%|          | 0/100 [00:00<?, ?it/s]


========== COMPLETE ==========
Tamil samples processed : 100
Saved to                : /content/tamil_pseudo_labels_100.csv

First 10 transcripts:

1. 24116.mp3
    இருவரும் சிறிது நேரம் மௌணமாகப் போய்க்கொண்டிருந்தார்கள்

2. 23214.mp3
    பாலம்பனின் செயலைத் தம் வயம் எடுத்துக்கொண்டன பெறுக்கெடுத்த குறுதியின் ஆவே

3. 17207.mp3
    தான் பார்த்தேனே மருதநாட்டிய சாஸ்கரத்தில் திருமாலின் முதல் அவதாரத்திற்கு ஒரு அஸ்தம் பிடித்தது

4. 23408.mp3
    எல்லோரும் தண்னைப் போலவே இருக்க வேண்டும் ென்றோ தன்னைப் புரிந்து கொள்ளவேன

5. 11602.mp3
    சசசக்கரங்கள் மண்னில் புதைந்து உருண்டன நெடும் பயணமாதலால்

6. 5756.mp3
    ியில் அசையும் பொருள் எதுவும் இல்லை நடுவில் நிற்பது சிலையாகப் புலப்பட்டது

7. 4552.mp3
    யோசித்துக் கொண்டிருக்கும்போது பழக்கத்தில் துவையலை விரல் வழித்து எடுத்துவிட்டது

8. 6350.mp3
    எல்லா உயிர்களுக்கும் அது தெரியும் அதனிடம் ஒப்படைத்து விட்டுப்போனாள் எந்த

9. 22640.mp3
    ்ு முக்க் புலப்பட்டதும் வந்தியத்தைவனுடைய பீதி அடியோடு அகன்றது ீ

10. 15639.mp3
    இருட்டுவதற்குள் இந்த இடம் வந்து சேரவ

In [17]:
import pandas as pd
import re
import unicodedata

print("========== CLEANING TAMIL PSEUDO-LABELS ==========")

TAMIL_PSEUDO_FILE = "/content/tamil_pseudo_labels_100.csv"
TAMIL_CLEAN_FILE = "/content/tamil_pseudo_labels_100_clean.csv"

df = pd.read_csv(TAMIL_PSEUDO_FILE)

print("Original samples:", len(df))


def tamil_ratio(text):

    if not isinstance(text, str):
        return 0.0

    chars = [
        ch for ch in text
        if not ch.isspace()
    ]

    if not chars:
        return 0.0

    tamil_chars = sum(
        1 for ch in chars
        if "\u0B80" <= ch <= "\u0BFF"
    )

    return tamil_chars / len(chars)


def has_repeated_characters(text):

    if not isinstance(text, str):
        return True

    # Detect 3 or more identical characters
    return bool(re.search(r"(.)\1{2,}", text))


def has_malformed_start(text):

    if not isinstance(text, str):
        return True

    text = text.strip()

    if not text:
        return True

    # Tamil combining marks should not normally appear
    # as the first character of a word/text.
    first = text[0]

    combining = unicodedata.category(first).startswith("M")

    return combining


clean_records = []

removed_empty = 0
removed_short = 0
removed_non_tamil = 0
removed_repeated = 0
removed_malformed = 0


for _, row in df.iterrows():

    text = str(row["text"]).strip()

    # 1. Empty
    if not text:
        removed_empty += 1
        continue

    # 2. Very short
    if len(text) < 5:
        removed_short += 1
        continue

    # 3. Tamil ratio
    ratio = tamil_ratio(text)

    if ratio < 0.80:
        removed_non_tamil += 1
        continue

    # 4. Repeated characters
    if has_repeated_characters(text):
        removed_repeated += 1
        continue

    # 5. Malformed starting character
    if has_malformed_start(text):
        removed_malformed += 1
        continue

    clean_records.append({
        "audio": row["audio"],
        "text": text,
        "language": "tam",
        "tamil_ratio": round(ratio, 3)
    })


clean_df = pd.DataFrame(clean_records)

clean_df.to_csv(
    TAMIL_CLEAN_FILE,
    index=False
)


print("\n========== CLEANING RESULT ==========")

print("Original samples        :", len(df))
print("Clean samples           :", len(clean_df))

print("\nRemoved:")
print("Empty                   :", removed_empty)
print("Too short               :", removed_short)
print("Non-Tamil               :", removed_non_tamil)
print("Repeated characters     :", removed_repeated)
print("Malformed start        :", removed_malformed)

print("\nSaved cleaned file:")
print(TAMIL_CLEAN_FILE)

print("\n========== CLEANED SAMPLE ==========")

for i, row in clean_df.head(20).iterrows():

    print(f"\n{i+1}. {row['text']}")
    print("   Tamil ratio:", row["tamil_ratio"])

print("\n====================================")

========== CLEANING TAMIL PSEUDO-LABELS ==========
Original samples: 100

========== CLEANING RESULT ==========
Original samples        : 100
Clean samples           : 82

Removed:
Empty                   : 0
Too short               : 1
Non-Tamil               : 0
Repeated characters     : 3
Malformed start        : 14

Saved cleaned file:
/content/tamil_pseudo_labels_100_clean.csv

========== CLEANED SAMPLE ==========

1. இருவரும் சிறிது நேரம் மௌணமாகப் போய்க்கொண்டிருந்தார்கள்
   Tamil ratio: 1.0

2. பாலம்பனின் செயலைத் தம் வயம் எடுத்துக்கொண்டன பெறுக்கெடுத்த குறுதியின் ஆவே
   Tamil ratio: 1.0

3. தான் பார்த்தேனே மருதநாட்டிய சாஸ்கரத்தில் திருமாலின் முதல் அவதாரத்திற்கு ஒரு அஸ்தம் பிடித்தது
   Tamil ratio: 1.0

4. எல்லோரும் தண்னைப் போலவே இருக்க வேண்டும் ென்றோ தன்னைப் புரிந்து கொள்ளவேன
   Tamil ratio: 1.0

5. யோசித்துக் கொண்டிருக்கும்போது பழக்கத்தில் துவையலை விரல் வழித்து எடுத்துவிட்டது
   Tamil ratio: 1.0

6. எல்லா உயிர்களுக்கும் அது தெரியும் அதனிடம் ஒப்படைத்து விட்டுப்போனாள் எந்த
   Tamil

In [18]:
print("========== TAMIL SUSPICIOUS TRANSCRIPT CHECK ==========")

import pandas as pd
import re

TAMIL_CLEAN_FILE = "/content/tamil_pseudo_labels_100_clean.csv"
df = pd.read_csv(TAMIL_CLEAN_FILE)

def repeated_word_ratio(text):
    words = text.split()

    if len(words) < 3:
        return 0.0

    counts = {}
    for word in words:
        counts[word] = counts.get(word, 0) + 1

    repeated_words = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return repeated_words / len(words)


def suspicious_transcript(text):
    words = text.split()

    if len(words) == 0:
        return True

    if repeated_word_ratio(text) > 0.30:
        return True

    for word in words:
        if len(word) > 30:
            return True

    return False


suspicious = []
good = []

for _, row in df.iterrows():

    text = str(row["text"]).strip()

    repeat_ratio = repeated_word_ratio(text)

    if suspicious_transcript(text):

        suspicious.append({
            "audio": row["audio"],
            "text": text,
            "repeat_ratio": round(repeat_ratio, 3)
        })

    else:

        good.append({
            "audio": row["audio"],
            "text": text,
            "language": "tam",
            "tamil_ratio": row["tamil_ratio"]
        })


good_df = pd.DataFrame(good)
suspicious_df = pd.DataFrame(suspicious)

print("\n========== RESULT ==========")
print("Good transcripts      :", len(good_df))
print("Suspicious transcripts:", len(suspicious_df))

print("\n========== SUSPICIOUS SAMPLES ==========")

for i, row in suspicious_df.head(20).iterrows():
    print(f"\n{i+1}. {row['text']}")
    print("   Repeated-word ratio:", row["repeat_ratio"])

print("\n========== GOOD SAMPLES ==========")

for i, row in good_df.head(10).iterrows():
    print(f"\n{i+1}. {row['text']}")

print("\n========================================")

========== TAMIL SUSPICIOUS TRANSCRIPT CHECK ==========

========== RESULT ==========
Good transcripts      : 81
Suspicious transcripts: 1

========== SUSPICIOUS SAMPLES ==========

1. கொண்டாட்டங்கள் கொண்டாட்டங்கள் கொண்டாட்டங்கள் கபிலரைத் தோழிலே சுமந்துவ
   Repeated-word ratio: 0.333

========== GOOD SAMPLES ==========

1. இருவரும் சிறிது நேரம் மௌணமாகப் போய்க்கொண்டிருந்தார்கள்

2. பாலம்பனின் செயலைத் தம் வயம் எடுத்துக்கொண்டன பெறுக்கெடுத்த குறுதியின் ஆவே

3. தான் பார்த்தேனே மருதநாட்டிய சாஸ்கரத்தில் திருமாலின் முதல் அவதாரத்திற்கு ஒரு அஸ்தம் பிடித்தது

4. எல்லோரும் தண்னைப் போலவே இருக்க வேண்டும் ென்றோ தன்னைப் புரிந்து கொள்ளவேன

5. யோசித்துக் கொண்டிருக்கும்போது பழக்கத்தில் துவையலை விரல் வழித்து எடுத்துவிட்டது

6. எல்லா உயிர்களுக்கும் அது தெரியும் அதனிடம் ஒப்படைத்து விட்டுப்போனாள் எந்த

7. இருட்டுவதற்குள் இந்த இடம் வந்து சேரவேண்டும் இடையில் நாகக் கிடங்கு வேரு

8. இப்ப்பதுபோல அமிக்டலாவுடன் மோதிப்பார்க்கும் இன்னொரு நபரும் நமக்குள்ளேயே

9. திருக்கும் நண்மைகள் பல் கிட்டின புதிய அறிமுகங்களும் தொடர

In [19]:

import pandas as pd
import shutil
import os

SOURCE_FILE = "/content/tamil_pseudo_labels_100_clean.csv"
BACKUP_FILE = "/content/tamil_pseudo_labels_backup_82.csv"

df = pd.read_csv(SOURCE_FILE)

# Remove the one suspicious repeated-word sample
def repeated_word_ratio(text):
    words = str(text).split()

    if len(words) < 3:
        return 0.0

    counts = {}

    for word in words:
        counts[word] = counts.get(word, 0) + 1

    repeated = sum(
        count - 1
        for count in counts.values()
        if count > 1
    )

    return repeated / len(words)


backup_df = df[
    df["text"].apply(repeated_word_ratio) <= 0.30
].copy()

backup_df.to_csv(
    BACKUP_FILE,
    index=False
)

print("========== TAMIL BACKUP ==========")
print("Original cleaned samples :", len(df))
print("Backup samples           :", len(backup_df))
print("Backup file              :", BACKUP_FILE)

print("\nLanguage:")
print(backup_df["language"].value_counts())

print("\n===================================")

========== TAMIL BACKUP ==========
Original cleaned samples : 82
Backup samples           : 81
Backup file              : /content/tamil_pseudo_labels_backup_82.csv

Language:
language
tam    81
Name: count, dtype: int64



In [20]:

import pandas as pd
import librosa
import torch
import os
from tqdm.auto import tqdm

print("========== GENERATING MALAYALAM PSEUDO-LABELS ==========")

MALAYALAM_PSEUDO_FILE = "/content/malayalam_pseudo_labels_100.csv"

# Use Malayalam adapter
processor.tokenizer.set_target_lang("mal")

malayalam_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="mal",
    ignore_mismatched_sizes=True
)

malayalam_model.load_adapter("mal")
malayalam_model = malayalam_model.to(device)
malayalam_model.eval()

records = []

# First 100 Malayalam files
selected_files = malayalam_files[:100]

for audio_file in tqdm(
    selected_files,
    desc="Malayalam transcription"
):

    try:

        speech, sr = librosa.load(
            audio_file,
            sr=16000,
            mono=True
        )

        inputs = processor(
            speech,
            sampling_rate=16000,
            return_tensors="pt"
        )

        input_values = inputs.input_values.to(device)

        with torch.no_grad():

            logits = malayalam_model(
                input_values
            ).logits

        predicted_ids = torch.argmax(
            logits,
            dim=-1
        )

        text = processor.batch_decode(
            predicted_ids
        )[0].strip()

        records.append({
            "audio": audio_file,
            "text": text,
            "language": "mal"
        })

    except Exception as e:

        print(
            "\nError:",
            os.path.basename(audio_file),
            "|",
            str(e)
        )

    # Save every 10 samples
    if len(records) % 10 == 0:

        pd.DataFrame(records).to_csv(
            MALAYALAM_PSEUDO_FILE,
            index=False
        )


# Final save
malayalam_pseudo_df = pd.DataFrame(records)

malayalam_pseudo_df.to_csv(
    MALAYALAM_PSEUDO_FILE,
    index=False
)

print("\n========== COMPLETE ==========")
print(
    "Malayalam samples processed :",
    len(malayalam_pseudo_df)
)

print(
    "Saved to                    :",
    MALAYALAM_PSEUDO_FILE
)

print("\nFirst 10 transcripts:")

for i, row in malayalam_pseudo_df.head(10).iterrows():

    print(
        f"\n{i+1}. {os.path.basename(row['audio'])}"
    )

    print("   ", row["text"])

print("==============================")

========== GENERATING MALAYALAM PSEUDO-LABELS ==========


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([136]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([136, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Malayalam transcription:   0%|          | 0/100 [00:00<?, ?it/s]


========== COMPLETE ==========
Malayalam samples processed : 100
Saved to                    : /content/malayalam_pseudo_labels_100.csv

First 10 transcripts:

1. 24116.mp3
    ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു

2. 23214.mp3
    സംശ്യാസ്പദമായ അംശയങ്ങൾ കാണാൻ കഴിയും രണ്ട് അഴ്ചമുമ്പ്

3. 17207.mp3
    പുതിയ തീവണ്ടിമുറി മെല്ലെ നീങ്ങുകയായിരുന്നു

4. 23408.mp3
    ഷക് നടക്കുകയാണ് എഴുന്നേറ്റ് അരുകിൽ ചെന്നപ്പോൾ അയാൾ ഒച്ച ഒതുക്കി പറിഞ്ഞു

5. 11602.mp3
    ഹോംസ് ദുഃഖത്തോടെ തല കുളിക്കി

6. 5756.mp3
    ഉറപ്പ് കൊടുക്കുന്തും മറ്റും കേട്ടതായി കുതിരവണ്ടിക്കാരൻ പറയുന്നു

7. 4552.mp3
    നേരിട് നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്നി

8. 6350.mp3
    ഒരു കൂട്ടം പേർ കടലിന്റെ തണുപ്പിലൂടെ നിന്തി്്

9. 22640.mp3
    അഞ്ചു മിനിറ്റ് കഴിഞ്ഞപ്പോഴേക്കും തൊട്ടുമുമ്പിൽ രണ്ട് പറ്ടാളക്കാർ കുതിരപ്പുറത്താണ്

10. 15639.mp3
    എന്നാലും ഒരു കാര്യും എല്ലാ പുസ്തകങ്ങളിലും എടുത്ത് പരഞ്ഞ്


In [21]:

import pandas as pd
import re
import unicodedata

print("========== CLEANING MALAYALAM PSEUDO-LABELS ==========")

MALAYALAM_PSEUDO_FILE = "/content/malayalam_pseudo_labels_100.csv"
MALAYALAM_CLEAN_FILE = "/content/malayalam_pseudo_labels_100_clean.csv"

df = pd.read_csv(MALAYALAM_PSEUDO_FILE)

print("Original samples:", len(df))


def malayalam_ratio(text):

    if not isinstance(text, str):
        return 0.0

    chars = [
        ch for ch in text
        if not ch.isspace()
    ]

    if not chars:
        return 0.0

    malayalam_chars = sum(
        1 for ch in chars
        if "\u0D00" <= ch <= "\u0D7F"
    )

    return malayalam_chars / len(chars)


def has_repeated_characters(text):

    if not isinstance(text, str):
        return True

    # Three or more identical characters
    return bool(
        re.search(r"(.)\1{2,}", text)
    )


def has_malformed_start(text):

    if not isinstance(text, str):
        return True

    text = text.strip()

    if not text:
        return True

    first = text[0]

    # Combining mark at the beginning
    return unicodedata.category(first).startswith("M")


clean_records = []

removed_empty = 0
removed_short = 0
removed_non_malayalam = 0
removed_repeated = 0
removed_malformed = 0


for _, row in df.iterrows():

    text = str(row["text"]).strip()

    # 1. Empty transcript
    if not text:
        removed_empty += 1
        continue

    # 2. Very short transcript
    if len(text) < 5:
        removed_short += 1
        continue

    # 3. Malayalam script check
    ratio = malayalam_ratio(text)

    if ratio < 0.80:
        removed_non_malayalam += 1
        continue

    # 4. Repeated characters
    if has_repeated_characters(text):
        removed_repeated += 1
        continue

    # 5. Malformed Unicode start
    if has_malformed_start(text):
        removed_malformed += 1
        continue

    clean_records.append({
        "audio": row["audio"],
        "text": text,
        "language": "mal",
        "malayalam_ratio": round(ratio, 3)
    })


clean_df = pd.DataFrame(clean_records)

clean_df.to_csv(
    MALAYALAM_CLEAN_FILE,
    index=False
)


print("\n========== CLEANING RESULT ==========")

print("Original samples        :", len(df))
print("Clean samples           :", len(clean_df))

print("\nRemoved:")
print("Empty                   :", removed_empty)
print("Too short              :", removed_short)
print("Non-Malayalam          :", removed_non_malayalam)
print("Repeated characters    :", removed_repeated)
print("Malformed start        :", removed_malformed)

print("\nSaved cleaned file:")
print(MALAYALAM_CLEAN_FILE)

print("\n========== CLEANED SAMPLE ==========")

for i, row in clean_df.head(20).iterrows():

    print(f"\n{i+1}. {row['text']}")
    print("   Malayalam ratio:", row["malayalam_ratio"])

print("\n====================================")

========== CLEANING MALAYALAM PSEUDO-LABELS ==========
Original samples: 100

========== CLEANING RESULT ==========
Original samples        : 100
Clean samples           : 87

Removed:
Empty                   : 0
Too short              : 1
Non-Malayalam          : 1
Repeated characters    : 2
Malformed start        : 9

Saved cleaned file:
/content/malayalam_pseudo_labels_100_clean.csv

========== CLEANED SAMPLE ==========

1. ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു
   Malayalam ratio: 1.0

2. സംശ്യാസ്പദമായ അംശയങ്ങൾ കാണാൻ കഴിയും രണ്ട് അഴ്ചമുമ്പ്
   Malayalam ratio: 1.0

3. പുതിയ തീവണ്ടിമുറി മെല്ലെ നീങ്ങുകയായിരുന്നു
   Malayalam ratio: 1.0

4. ഷക് നടക്കുകയാണ് എഴുന്നേറ്റ് അരുകിൽ ചെന്നപ്പോൾ അയാൾ ഒച്ച ഒതുക്കി പറിഞ്ഞു
   Malayalam ratio: 1.0

5. ഹോംസ് ദുഃഖത്തോടെ തല കുളിക്കി
   Malayalam ratio: 1.0

6. ഉറപ്പ് കൊടുക്കുന്തും മറ്റും കേട്ടതായി കുതിരവണ്ടിക്കാരൻ പറയുന്നു
   Malayalam ratio: 1.0

7. നേരിട് നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്നി
   Malayalam ratio: 1.0

8. ഒരു 

In [22]:
import pandas as pd
import re

print("========== MALAYALAM SUSPICIOUS TRANSCRIPT CHECK ==========")

MALAYALAM_CLEAN_FILE = "/content/malayalam_pseudo_labels_100_clean.csv"

df = pd.read_csv(MALAYALAM_CLEAN_FILE)

print("Clean samples:", len(df))


def repeated_word_ratio(text):

    words = str(text).split()

    if len(words) < 2:
        return 0.0

    repeated = 0

    for i in range(1, len(words)):
        if words[i] == words[i - 1]:
            repeated += 1

    return repeated / len(words)


def suspicious_unicode(text):

    # Detect unusual repeated combining marks
    return bool(
        re.search(r"[\u0D00-\u0D7F]{0}", "")
    )


suspicious_records = []
good_records = []

for _, row in df.iterrows():

    text = str(row["text"]).strip()

    words = text.split()

    repeat_ratio = repeated_word_ratio(text)

    # Flag extremely short or repeated-word outputs
    suspicious = False

    if len(words) < 2:
        suspicious = True

    if repeat_ratio >= 0.30:
        suspicious = True

    # Flag obvious repeated character patterns
    if re.search(r"(.)\1{2,}", text):
        suspicious = True

    record = {
        "audio": row["audio"],
        "text": text,
        "language": "mal",
        "word_count": len(words),
        "repeated_word_ratio": round(repeat_ratio, 3)
    }

    if suspicious:
        suspicious_records.append(record)
    else:
        good_records.append(record)


suspicious_df = pd.DataFrame(suspicious_records)
good_df = pd.DataFrame(good_records)


SUSPICIOUS_FILE = "/content/malayalam_pseudo_labels_suspicious.csv"
GOOD_FILE = "/content/malayalam_pseudo_labels_backup.csv"

suspicious_df.to_csv(
    SUSPICIOUS_FILE,
    index=False
)

good_df.to_csv(
    GOOD_FILE,
    index=False
)


print("\n========== RESULT ==========")

print("Total clean samples     :", len(df))
print("Good samples             :", len(good_df))
print("Suspicious samples       :", len(suspicious_df))

print("\nSaved:")
print("Good      :", GOOD_FILE)
print("Suspicious:", SUSPICIOUS_FILE)


print("\n========== SUSPICIOUS SAMPLES ==========")

if len(suspicious_df) == 0:

    print("No suspicious samples detected.")

else:

    for i, row in suspicious_df.iterrows():

        print(f"\n{i+1}. {row['text']}")
        print("   Words:", row["word_count"])
        print(
            "   Repeated-word ratio:",
            row["repeated_word_ratio"]
        )


print("\n========================================")

========== MALAYALAM SUSPICIOUS TRANSCRIPT CHECK ==========
Clean samples: 87

========== RESULT ==========
Total clean samples     : 87
Good samples             : 87
Suspicious samples       : 0

Saved:
Good      : /content/malayalam_pseudo_labels_backup.csv
Suspicious: /content/malayalam_pseudo_labels_suspicious.csv

========== SUSPICIOUS SAMPLES ==========
No suspicious samples detected.



In [23]:
from datasets import Dataset, Audio
import pandas as pd

print("========== PREPARING MALAYALAM DATA ==========")

MALAYALAM_DATA_FILE = "/content/malayalam_pseudo_labels_backup.csv"

mal_df = pd.read_csv(MALAYALAM_DATA_FILE)

print("Samples loaded:", len(mal_df))
print("\nColumns:")
print(mal_df.columns.tolist())

print("\nLanguage distribution:")
print(mal_df["language"].value_counts())

# Keep only required columns
mal_df = mal_df[
    ["audio", "text", "language"]
].copy()

# Create Hugging Face Dataset
mal_dataset = Dataset.from_pandas(
    mal_df,
    preserve_index=False
)

# Convert audio to 16 kHz
mal_dataset = mal_dataset.cast_column(
    "audio",
    Audio(sampling_rate=16000)
)

print("\n========== DATASET READY ==========")

print("Number of samples:", len(mal_dataset))
print("Features:", mal_dataset.features)

print("\nFirst sample:")
print("Text:", mal_dataset[0]["text"])
print("Language:", mal_dataset[0]["language"])

audio_info = mal_dataset[0]["audio"]

print("Audio object:", type(audio_info))
print("Sample rate:", audio_info["sampling_rate"])

print("\n====================================")

========== PREPARING MALAYALAM DATA ==========
Samples loaded: 87

Columns:
['audio', 'text', 'language', 'word_count', 'repeated_word_ratio']

Language distribution:
language
mal    87
Name: count, dtype: int64

========== DATASET READY ==========
Number of samples: 87
Features: {'audio': Audio(sampling_rate=16000, decode=True, stream_index=None), 'text': Value('string'), 'language': Value('string')}

First sample:
Text: ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു
Language: mal
Audio object: <class 'datasets.features._torchcodec.AudioDecoder'>
Sample rate: 16000



In [24]:
from transformers import AutoProcessor

print("========== CONFIGURING MALAYALAM MMS ==========")

MODEL_NAME = "facebook/mms-1b-all"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME
)

# Select Malayalam vocabulary
processor.tokenizer.set_target_lang("mal")

print("Model:", MODEL_NAME)
print("Target language: Malayalam (mal)")

print("\nTokenizer vocabulary size:")
print(len(processor.tokenizer))

print("\nProcessor configured successfully!")
print("===============================================")

========== CONFIGURING MALAYALAM MMS ==========
Model: facebook/mms-1b-all
Target language: Malayalam (mal)

Tokenizer vocabulary size:
136

Processor configured successfully!


In [25]:
print("========== PREPROCESSING MALAYALAM DATA ==========")

def prepare_malayalam_dataset(batch):

    # Get audio
    audio = batch["audio"]

    # Audio waveform
    speech = audio["array"]

    # Convert audio into MMS input features
    audio_inputs = processor(
        speech,
        sampling_rate=16000
    )

    batch["input_values"] = audio_inputs.input_values[0]

    # Convert Malayalam transcript into token IDs
    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch


mal_processed = mal_dataset.map(
    prepare_malayalam_dataset,
    remove_columns=mal_dataset.column_names,
    desc="Processing Malayalam audio"
)

print("\n========== PREPROCESSING COMPLETE ==========")

print("Processed samples:", len(mal_processed))
print("Features:", mal_processed.features)

print("\nFirst sample:")
print("Input values length:", len(mal_processed[0]["input_values"]))
print("Labels:", mal_processed[0]["labels"])

print("\n=============================================")

========== PREPROCESSING MALAYALAM DATA ==========


Processing Malayalam audio:   0%|          | 0/87 [00:00<?, ? examples/s]


========== PREPROCESSING COMPLETE ==========
Processed samples: 87
Features: {'input_values': List(Value('float32')), 'labels': List(Value('int64'))}

First sample:
Input values length: 79638
Labels: [10, 4, 7, 16, 7, 9, 19, 6, 24, 4, 24, 4, 5, 45, 20, 27, 41, 6, 7, 4, 7, 11, 19, 6, 20, 4, 20, 11, 12, 6, 29, 5, 7, 17, 20, 4, 20, 18, 9, 8, 4, 4, 5, 10, 6, 7, 12, 11, 21, 5, 14, 27, 7, 9, 8, 4, 8, 9]



In [26]:
print("========== VERIFYING MALAYALAM LABELS ==========")

sample_labels = mal_processed[0]["labels"]

decoded_text = processor.tokenizer.decode(
    sample_labels
)

original_text = mal_dataset[0]["text"]

print("Original transcript:")
print(original_text)

print("\nDecoded transcript:")
print(decoded_text)

print("\nLabel token count:", len(sample_labels))

print("\nExact match:", original_text == decoded_text)

print("===============================================")

========== VERIFYING MALAYALAM LABELS ==========
Original transcript:
ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു

Decoded transcript:
ത്കെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന് തികയാം പോകുന്നു

Label token count: 58

Exact match: False


In [27]:
from dataclasses import dataclass
from typing import Union
import torch

print("========== CREATING CTC DATA COLLATOR ==========")

@dataclass
class DataCollatorCTCWithPadding:
    processor: object
    padding: Union[bool, str] = True

    def __call__(self, features):

        # Audio inputs
        input_features = [
            {
                "input_values": feature["input_values"]
            }
            for feature in features
        ]

        # Transcript labels
        label_features = [
            {
                "input_ids": feature["labels"]
            }
            for feature in features
        ]

        # Pad audio
        batch = self.processor.pad(
            input_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Pad labels
        labels_batch = self.processor.tokenizer.pad(
            label_features,
            padding=self.padding,
            return_tensors="pt"
        )

        # Ignore padded labels during CTC loss
        labels = labels_batch["input_ids"].masked_fill(
            labels_batch.attention_mask.ne(1),
            -100
        )

        batch["labels"] = labels

        return batch


data_collator = DataCollatorCTCWithPadding(
    processor=processor,
    padding=True
)

print("CTC data collator created successfully!")

print("===============================================")

========== CREATING CTC DATA COLLATOR ==========
CTC data collator created successfully!


In [28]:
print("========== SPLITTING MALAYALAM DATA ==========")

# Shuffle the dataset
mal_dataset = mal_dataset.shuffle(seed=42)

# 80% training / 20% validation
split = mal_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

mal_train_raw = split["train"]
mal_val_raw = split["test"]

print("Total samples      :", len(mal_dataset))
print("Training samples   :", len(mal_train_raw))
print("Validation samples :", len(mal_val_raw))

print("\nTraining sample:")
print(mal_train_raw[0]["text"])

print("\nValidation sample:")
print(mal_val_raw[0]["text"])

print("\n==============================================")

========== SPLITTING MALAYALAM DATA ==========
Total samples      : 87
Training samples   : 69
Validation samples : 18

Training sample:
പക്ഷേ അല്പ കാലം നുമ്പ് ഞങ്ങളുടെ വിശ്വസ്തനായ ഒരു സൂഹൃത്ത്

Validation sample:
നേരിട് നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്നി



In [29]:
print("========== PREPROCESSING TRAIN/VALIDATION DATA ==========")

def prepare_malayalam_dataset(batch):

    audio = batch["audio"]

    speech = audio["array"]

    audio_inputs = processor(
        speech,
        sampling_rate=16000
    )

    batch["input_values"] = audio_inputs.input_values[0]

    batch["labels"] = processor.tokenizer(
        batch["text"]
    ).input_ids

    return batch


mal_train = mal_train_raw.map(
    prepare_malayalam_dataset,
    remove_columns=mal_train_raw.column_names,
    desc="Processing Malayalam training data"
)

mal_val = mal_val_raw.map(
    prepare_malayalam_dataset,
    remove_columns=mal_val_raw.column_names,
    desc="Processing Malayalam validation data"
)

print("\n========== PREPROCESSING COMPLETE ==========")

print("Training samples   :", len(mal_train))
print("Validation samples :", len(mal_val))

print("\nTraining features:")
print(mal_train.features)

print("\nValidation features:")
print(mal_val.features)

print("\nFirst training sample:")
print("Input length:", len(mal_train[0]["input_values"]))
print("Label length:", len(mal_train[0]["labels"]))

print("\n============================================")

========== PREPROCESSING TRAIN/VALIDATION DATA ==========


Processing Malayalam training data:   0%|          | 0/69 [00:00<?, ? examples/s]

Processing Malayalam validation data:   0%|          | 0/18 [00:00<?, ? examples/s]


========== PREPROCESSING COMPLETE ==========
Training samples   : 69
Validation samples : 18

Training features:
{'input_values': List(Value('float32')), 'labels': List(Value('int64'))}

Validation features:
{'input_values': List(Value('float32')), 'labels': List(Value('int64'))}

First training sample:
Input length: 80080
Label length: 56



In [31]:
import torch
import gc
from transformers import Wav2Vec2ForCTC

print("========== CLEARING GPU MEMORY ==========")

# Delete previous models if they exist
for variable_name in ["model", "mal_model"]:
    if variable_name in globals():
        del globals()[variable_name]

# Python garbage collection
gc.collect()

# Clear unused CUDA memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "Free GPU memory before loading:",
        round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
        "GB"
    )

print("\n========== LOADING MALAYALAM MMS FP16 ==========")

MODEL_NAME = "facebook/mms-1b-all"

mal_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_NAME,
    target_lang="mal",
    ignore_mismatched_sizes=True,
    torch_dtype=torch.float16
)

mal_model.load_adapter("mal")

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

mal_model = mal_model.to(device)

print("\n========== MODEL LOADED ==========")

print("Language adapter :", "Malayalam (mal)")
print("Device           :", device)
print("Data type        :", next(mal_model.parameters()).dtype)

total_params = sum(
    p.numel() for p in mal_model.parameters()
)

print("Total parameters :", total_params)

print(
    "Free GPU memory after loading:",
    round(torch.cuda.mem_get_info()[0] / 1024**3, 2),
    "GB"
)

print("============================================")

========== CLEARING GPU MEMORY ==========


`torch_dtype` is deprecated! Use `dtype` instead!


GPU: Tesla T4
Free GPU memory before loading: 3.96 GB

========== LOADING MALAYALAM MMS FP16 ==========


Some weights of Wav2Vec2ForCTC were not initialized from the model checkpoint at facebook/mms-1b-all and are newly initialized because the shapes did not match:
- lm_head.bias: found shape torch.Size([154]) in the checkpoint and torch.Size([136]) in the model instantiated
- lm_head.weight: found shape torch.Size([154, 1280]) in the checkpoint and torch.Size([136, 1280]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



========== MODEL LOADED ==========
Language adapter : Malayalam (mal)
Device           : cuda
Data type        : torch.float16
Total parameters : 964822792
Free GPU memory after loading: 2.01 GB


In [32]:
print("========== CONFIGURING MALAYALAM TRAINING ==========")

# Freeze the entire MMS model first
for param in mal_model.parameters():
    param.requires_grad = False

# Train only Malayalam adapter layers
adapter_params = 0

for name, param in mal_model.named_parameters():
    if "adapter_layer" in name:
        param.requires_grad = True
        adapter_params += param.numel()

# Train Malayalam language head
lm_head_params = 0

for name, param in mal_model.named_parameters():
    if "lm_head" in name:
        param.requires_grad = True
        lm_head_params += param.numel()

# Enable gradient checkpointing to reduce GPU memory
mal_model.gradient_checkpointing_enable()

# Required when using gradient checkpointing
mal_model.config.use_cache = False

# Count trainable parameters
trainable_params = sum(
    p.numel()
    for p in mal_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in mal_model.parameters()
)

print("\n========== TRAINING CONFIGURATION ==========")

print("Total parameters     :", total_params)
print("Adapter parameters   :", adapter_params)
print("LM head parameters   :", lm_head_params)
print("Trainable parameters :", trainable_params)

print(
    "Trainable percentage  :",
    round((trainable_params / total_params) * 100, 4),
    "%"
)

print("\nTrainable components:")
print(" - Malayalam adapter layers")
print(" - Malayalam LM head")
print(" - Base MMS model frozen")
print(" - Gradient checkpointing enabled")
print(" - FP16 enabled")

print("\n============================================")

========== CONFIGURING MALAYALAM TRAINING ==========

========== TRAINING CONFIGURATION ==========
Total parameters     : 964822792
Adapter parameters   : 2151168
LM head parameters   : 174216
Trainable parameters : 2325384
Trainable percentage  : 0.241 %

Trainable components:
 - Malayalam adapter layers
 - Malayalam LM head
 - Base MMS model frozen
 - Gradient checkpointing enabled
 - FP16 enabled



In [33]:
print("========== CHECKING MALAYALAM TRAINING BATCH ==========")

# Take 2 samples for a memory-safe sanity check
sample_features = [
    mal_train[i]
    for i in range(2)
]

batch = data_collator(sample_features)

print("\nBatch keys:")
print(batch.keys())

print("\nInput values shape:")
print(batch["input_values"].shape)

print("\nLabels shape:")
print(batch["labels"].shape)

print("\nInput dtype:")
print(batch["input_values"].dtype)

print("\nLabels dtype:")
print(batch["labels"].dtype)

print("\nNumber of ignored label positions (-100):")
print((batch["labels"] == -100).sum().item())

print("\n========== BATCH CHECK COMPLETE ==========")

========== CHECKING MALAYALAM TRAINING BATCH ==========

Batch keys:
KeysView({'input_values': tensor([[-8.7609e-06, -8.7609e-06, -8.7609e-06,  ...,  7.4701e-02,
          6.2466e-02,  5.4240e-02],
        [ 1.7508e-04,  1.7508e-04,  1.7508e-04,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32), 'labels': tensor([[  14,    7,    4,   36,   30,    5,   31,   20,    4,   14,    5,    7,
           11,   20,   21,    5,    8,    9,   18,    4,   14,    4,    5,   46,
           25,    4,   25,   23,    9,   15,   16,    5,   17,    6,   38,    4,
           17,   22,    4,   10,    8,   11,   12,    5,   44,   13,    9,    5,
           22,   39,   50,   65,   10,    4,   10,    4, -100, -100, -100, -100],
        [   7,    5,   11,   29,   11,   17,    4,    5,   48,   11,   13,    6,
            7,    4,    7,    9,   23,    4,   23,    5,   49,   14,   50,   11,
           1

In [34]:
from transformers import TrainingArguments

print("========== CONFIGURING MALAYALAM TRAINING ARGUMENTS ==========")

training_args = TrainingArguments(
    output_dir="/content/eldercare_voice_ai/malayalam_mms_model",

    # Training
    num_train_epochs=3,
    learning_rate=1e-4,

    # T4 memory-safe settings
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    # Evaluation & saving
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,

    # Performance
    fp16=True,
    gradient_checkpointing=True,

    # Logging
    logging_steps=5,
    report_to="none",

    # Avoid multiprocessing memory problems
    dataloader_num_workers=0,

    # Disable unnecessary column removal
    remove_unused_columns=False
)

print("\n========== TRAINING ARGUMENTS READY ==========")

print("Epochs                  :", training_args.num_train_epochs)
print("Learning rate           :", training_args.learning_rate)
print("Train batch size        :", training_args.per_device_train_batch_size)
print("Gradient accumulation   :", training_args.gradient_accumulation_steps)
print("Effective batch size    :",
      training_args.per_device_train_batch_size *
      training_args.gradient_accumulation_steps)
print("FP16                    :", training_args.fp16)
print("Gradient checkpointing  :", training_args.gradient_checkpointing)
print("Evaluation              :", training_args.eval_strategy)
print("Saving                  :", training_args.save_strategy)

print("==============================================")

========== CONFIGURING MALAYALAM TRAINING ARGUMENTS ==========

========== TRAINING ARGUMENTS READY ==========
Epochs                  : 3
Learning rate           : 0.0001
Train batch size        : 1
Gradient accumulation   : 8
Effective batch size    : 8
FP16                    : True
Gradient checkpointing  : True
Evaluation              : IntervalStrategy.EPOCH
Saving                  : SaveStrategy.EPOCH


In [35]:
from transformers import Trainer

print("========== CREATING MALAYALAM TRAINER ==========")

mal_trainer = Trainer(
    model=mal_model,
    args=training_args,
    train_dataset=mal_train,
    eval_dataset=mal_val,
    data_collator=data_collator
)

print("\nMalayalam Trainer created successfully!")

print("Training samples   :", len(mal_train))
print("Validation samples :", len(mal_val))
print("Trainable params   :", sum(
    p.numel()
    for p in mal_model.parameters()
    if p.requires_grad
))

print("===============================================")

========== CREATING MALAYALAM TRAINER ==========

Malayalam Trainer created successfully!
Training samples   : 69
Validation samples : 18
Trainable params   : 2325384


In [38]:
import torch

print("========== FIXING FP16 GRADIENT CONFIGURATION ==========")

# Keep the model on GPU but convert trainable parameters
# to FP32 so AMP can safely manage their gradients.

trainable_count = 0

for name, param in mal_model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()
        trainable_count += param.numel()

print("Trainable parameters converted to FP32:", trainable_count)

# Verify trainable parameter dtypes
print("\nTrainable parameter dtypes:")

dtypes = {}

for name, param in mal_model.named_parameters():
    if param.requires_grad:
        dtypes[str(param.dtype)] = dtypes.get(str(param.dtype), 0) + 1

for dtype, count in dtypes.items():
    print(" -", dtype, ":", count, "tensors")

print("\nFP16 AMP remains enabled in Trainer:", training_args.fp16)

print("\n========== FIX COMPLETE ==========")

========== FIXING FP16 GRADIENT CONFIGURATION ==========
Trainable parameters converted to FP32: 2325384

Trainable parameter dtypes:
 - torch.float32 : 290 tensors

FP16 AMP remains enabled in Trainer: True

========== FIX COMPLETE ==========


In [39]:
from transformers import Trainer

print("========== RECREATING MALAYALAM TRAINER ==========")

mal_trainer = Trainer(
    model=mal_model,
    args=training_args,
    train_dataset=mal_train,
    eval_dataset=mal_val,
    data_collator=data_collator
)

print("\nMalayalam Trainer recreated successfully!")

print("Training samples   :", len(mal_train))
print("Validation samples :", len(mal_val))

trainable_params = sum(
    p.numel()
    for p in mal_model.parameters()
    if p.requires_grad
)

print("Trainable params   :", trainable_params)

print("\nTrainable parameter dtype check:")

trainable_dtypes = set(
    str(p.dtype)
    for p in mal_model.parameters()
    if p.requires_grad
)

for dtype in trainable_dtypes:
    print(" -", dtype)

print("\nFP16 AMP:", training_args.fp16)
print("Gradient checkpointing:", training_args.gradient_checkpointing)

print("\n========== TRAINER READY ==========")

========== RECREATING MALAYALAM TRAINER ==========

Malayalam Trainer recreated successfully!
Training samples   : 69
Validation samples : 18
Trainable params   : 2325384

Trainable parameter dtype check:
 - torch.float32

FP16 AMP: True
Gradient checkpointing: True

========== TRAINER READY ==========


In [40]:
print("========== STARTING MALAYALAM TRAINING ==========")

print("Training samples   :", len(mal_train))
print("Validation samples :", len(mal_val))
print("Epochs             :", training_args.num_train_epochs)

print("\nTraining started...")
print("===============================================")

mal_train_result = mal_trainer.train()

print("\n========== MALAYALAM TRAINING COMPLETE ==========")

print("Training completed successfully!")

print("\nTraining metrics:")
print(mal_train_result.metrics)

print("================================================")

========== STARTING MALAYALAM TRAINING ==========
Training samples   : 69
Validation samples : 18
Epochs             : 3

Training started...


Epoch,Training Loss,Validation Loss
1,0.964300,0.299178
2,0.746300,0.278088
3,0.752200,0.270458



========== MALAYALAM TRAINING COMPLETE ==========
Training completed successfully!

Training metrics:
{'train_runtime': 262.3017, 'train_samples_per_second': 0.789, 'train_steps_per_second': 0.103, 'total_flos': 9.562379338719749e+16, 'train_loss': 0.8369386461046007, 'epoch': 3.0}


In [41]:
print("========== SAVING MALAYALAM MODEL ==========")

MALAYALAM_MODEL_PATH = (
    "/content/eldercare_voice_ai/malayalam_mms_model/final"
)

mal_trainer.save_model(MALAYALAM_MODEL_PATH)
processor.save_pretrained(MALAYALAM_MODEL_PATH)

print("\nMalayalam model saved successfully!")

print("Model path:")
print(MALAYALAM_MODEL_PATH)

print("\n========== SAVE COMPLETE ==========")

========== SAVING MALAYALAM MODEL ==========

Malayalam model saved successfully!
Model path:
/content/eldercare_voice_ai/malayalam_mms_model/final

========== SAVE COMPLETE ==========


In [42]:
import torch
import librosa
from transformers import AutoProcessor, Wav2Vec2ForCTC

print("========== TESTING TRAINED MALAYALAM MODEL ==========")

MODEL_PATH = "/content/eldercare_voice_ai/malayalam_mms_model/final"

# Load trained processor
trained_processor = AutoProcessor.from_pretrained(
    MODEL_PATH
)

trained_processor.tokenizer.set_target_lang("mal")

# Load trained model
trained_model = Wav2Vec2ForCTC.from_pretrained(
    MODEL_PATH
)

trained_model = trained_model.to(device)
trained_model.eval()

# Test on a Malayalam audio file
test_file = malayalam_files[0]

print("Test audio:", test_file)

speech, sr = librosa.load(
    test_file,
    sr=16000,
    mono=True
)

print("Sample rate:", sr)
print("Duration:", round(len(speech) / sr, 2), "seconds")

# Prepare audio
inputs = trained_processor(
    speech,
    sampling_rate=16000,
    return_tensors="pt"
)

input_values = inputs.input_values.to(device)

# Inference
with torch.no_grad():
    logits = trained_model(input_values).logits

predicted_ids = torch.argmax(
    logits,
    dim=-1
)

transcription = trained_processor.batch_decode(
    predicted_ids
)[0]

print("\n========== TRAINED MODEL RESULT ==========")
print("Malayalam transcription:")
print(transcription)

print("\n==========================================")

========== TESTING TRAINED MALAYALAM MODEL ==========
Test audio: /root/.cache/kagglehub/datasets/hmsolanki/indian-languages-audio-dataset/versions/1/Indian_Languages_Audio_Dataset/Malayalam/24116.mp3
Sample rate: 16000
Duration: 4.98 seconds

========== TRAINED MODEL RESULT ==========
Malayalam transcription:
ത്തെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു



In [43]:
import librosa
import torch

print("========== TESTING TRAINED MALAYALAM MODEL ==========")

# Use 10 Malayalam samples
test_files = malayalam_files[:10]

for i, test_file in enumerate(test_files, start=1):

    speech, sr = librosa.load(
        test_file,
        sr=16000,
        mono=True
    )

    inputs = trained_processor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    )

    input_values = inputs.input_values.to(device)

    with torch.no_grad():
        logits = trained_model(input_values).logits

    predicted_ids = torch.argmax(
        logits,
        dim=-1
    )

    transcription = trained_processor.batch_decode(
        predicted_ids
    )[0]

    print(f"\n{i}. {os.path.basename(test_file)}")
    print("   Prediction:", transcription)

print("\n========== 10-SAMPLE TEST COMPLETE ==========")

========== TESTING TRAINED MALAYALAM MODEL ==========

1. 24116.mp3
   Prediction: ത്തെകുറിച്ച് ആലോജിക്കാറില്ലായിർ കവല്ലമുന്് തികയാം പോകുന്നു

2. 23214.mp3
   Prediction: സംശയാസ്പതമായ അംശയങ്ങൾ കാണാൻ കഴിയും രണ്ട് അഴ്ചമുമ്പ്

3. 17207.mp3
   Prediction: പുതിയ തീവണ്ടിമുറി മെല്ലെ നീങ്ങുകയായിരുന്നു

4. 23408.mp3
   Prediction: ് നടക്കുകയാണ് എഴുന്നേറ്റ് അരുകിൽ ചെന്നപ്പോൾ അയാൾ ഒച്ച ഒതുക്കി പറിഞ്ഞു

5. 11602.mp3
   Prediction: ഹോംസ് ദുഃഖത്തോടെ തല കുളിക്കി

6. 5756.mp3
   Prediction: ഉറപ്പ് കൊടുക്കുന്തും മറ്റും കേട്ടതായി കുതിരവണ്ടിക്കാരൻ പറയുന്നു

7. 4552.mp3
   Prediction: നേരിടു നാപത്ത് വാദില്ലൂടെയോ ജനല്ലൂടെയോ ല്ല എന്നി

8. 6350.mp3
   Prediction: ഒരു കൂട്ടംപേർ കടലിന്റെ തണുപ്പിലൂടെ നന്തി്

9. 22640.mp3
   Prediction: അഞ്ചു മിനിറ്റു കഴിഞ്ഞപ്പോഴേക്കും തൊട്ടുമുമ്പിൽ രണ്ട് പറ്ടാളക്കാർ കുതിരപ്പുറത്താണ്

10. 15639.mp3
   Prediction: എന്നാലും ഒരു കാര്യും എല്ലാ പുസ്തകങ്ങളിലും എടുത്ത് പരഞ്ഞ

========== 10-SAMPLE TEST COMPLETE ==========


In [44]:
print("========== CHECKING TRAINED MODEL WEIGHTS ==========")

# Count trainable parameters
trainable_params = [
    p for p in mal_model.parameters()
    if p.requires_grad
]

print("Trainable tensors:", len(trainable_params))

# Check whether gradients exist
grad_tensors = [
    p for p in trainable_params
    if p.grad is not None
]

print("Tensors with gradients:", len(grad_tensors))

# Calculate gradient information
if grad_tensors:
    total_grad = sum(
        p.grad.abs().mean().item()
        for p in grad_tensors
    )

    print("Average gradient measure:", total_grad / len(grad_tensors))
else:
    print("No gradients currently stored.")

print("\n========== MODEL WEIGHT CHECK COMPLETE ==========")

========== CHECKING TRAINED MODEL WEIGHTS ==========
Trainable tensors: 290
Tensors with gradients: 0
No gradients currently stored.

========== MODEL WEIGHT CHECK COMPLETE ==========


In [45]:
import os

print("========== CHECKING MALAYALAM TRAINING CHECKPOINTS ==========")

OUTPUT_DIR = "/content/eldercare_voice_ai/malayalam_mms_model"

if os.path.exists(OUTPUT_DIR):

    print("Training directory exists:")
    print(OUTPUT_DIR)

    print("\nFiles and folders:")

    for item in sorted(os.listdir(OUTPUT_DIR)):
        path = os.path.join(OUTPUT_DIR, item)

        if os.path.isdir(path):
            print("📁", item)
        else:
            print("📄", item)

else:
    print("Training directory NOT FOUND!")

print("\n========== CHECK COMPLETE ==========")

========== CHECKING MALAYALAM TRAINING CHECKPOINTS ==========
Training directory exists:
/content/eldercare_voice_ai/malayalam_mms_model

Files and folders:
📁 checkpoint-27
📁 final

========== CHECK COMPLETE ==========


In [46]:
import os

CHECKPOINT_PATH = (
    "/content/eldercare_voice_ai/malayalam_mms_model/checkpoint-27"
)

print("========== INSPECTING TRAINED CHECKPOINT ==========")

print("Checkpoint path:")
print(CHECKPOINT_PATH)

if os.path.exists(CHECKPOINT_PATH):

    print("\nCheckpoint contents:")

    for item in sorted(os.listdir(CHECKPOINT_PATH)):
        path = os.path.join(CHECKPOINT_PATH, item)

        if os.path.isdir(path):
            print("📁", item)
        else:
            size_mb = os.path.getsize(path) / (1024 * 1024)
            print(f"📄 {item} ({size_mb:.2f} MB)")

else:
    print("Checkpoint NOT FOUND!")

print("\n========== CHECKPOINT INSPECTION COMPLETE ==========")

========== INSPECTING TRAINED CHECKPOINT ==========
Checkpoint path:
/content/eldercare_voice_ai/malayalam_mms_model/checkpoint-27

Checkpoint contents:
📄 config.json (0.00 MB)
📄 model.safetensors (1844.82 MB)
📄 optimizer.pt (17.98 MB)
📄 rng_state.pth (0.01 MB)
📄 scaler.pt (0.00 MB)
📄 scheduler.pt (0.00 MB)
📄 trainer_state.json (0.00 MB)
📄 training_args.bin (0.01 MB)

========== CHECKPOINT INSPECTION COMPLETE ==========


In [47]:
import torch
from safetensors.torch import load_file

print("========== VERIFYING TRAINED WEIGHTS ==========")

CHECKPOINT_PATH = (
    "/content/eldercare_voice_ai/malayalam_mms_model/checkpoint-27"
)

WEIGHTS_FILE = CHECKPOINT_PATH + "/model.safetensors"

print("Loading checkpoint weights...")

checkpoint_state = load_file(
    WEIGHTS_FILE,
    device="cpu"
)

print("Checkpoint tensors:", len(checkpoint_state))

# Find trainable parameters
trainable_names = [
    name
    for name, param in mal_model.named_parameters()
    if param.requires_grad
]

print("Trainable model tensors:", len(trainable_names))

# Check whether the trained tensors exist in checkpoint
found = 0

for name in trainable_names:
    if name in checkpoint_state:
        found += 1

print("Trainable tensors found in checkpoint:", found)

print("\n========== RESULT ==========")

if found == len(trainable_names):
    print("✅ All trainable parameters are present in the checkpoint.")
else:
    print("⚠️ Some trainable parameters were not found.")

print("\nCheckpoint weight file size:")
print(
    round(
        os.path.getsize(WEIGHTS_FILE) / (1024**3),
        2
    ),
    "GB"
)

print("\n========== WEIGHT VERIFICATION COMPLETE ==========")

========== VERIFYING TRAINED WEIGHTS ==========
Loading checkpoint weights...
Checkpoint tensors: 1096
Trainable model tensors: 290
Trainable tensors found in checkpoint: 290

========== RESULT ==========
✅ All trainable parameters are present in the checkpoint.

Checkpoint weight file size:
1.8 GB

========== WEIGHT VERIFICATION COMPLETE ==========


In [49]:
from transformers import AutoProcessor, Wav2Vec2ForCTC

MODEL_PATH = "/content/eldercare_voice_ai/malayalam_mms_model/final"

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = Wav2Vec2ForCTC.from_pretrained(MODEL_PATH)

print("========== TRAINED LANGUAGE ==========")

print("Model path:", MODEL_PATH)

# Tokenizer information
print("Tokenizer language:", processor.tokenizer.target_lang)

# Vocabulary size
print("Vocabulary size:", len(processor.tokenizer))

# Model language adapter
if hasattr(model, "target_lang"):
    print("Model target language:", model.target_lang)

print("======================================")

========== TRAINED LANGUAGE ==========
Model path: /content/eldercare_voice_ai/malayalam_mms_model/final
Tokenizer language: mal
Vocabulary size: 136
Model target language: None


In [50]:
import json
import os

MODEL_PATH = "/content/eldercare_voice_ai/malayalam_mms_model/final"

print("========== CHECKING SAVED MALAYALAM MODEL ==========")

config_path = os.path.join(MODEL_PATH, "config.json")

with open(config_path, "r") as f:
    config = json.load(f)

print("Model type:", config.get("model_type"))
print("Vocab size:", config.get("vocab_size"))

print("\nModel directory contents:")
for item in sorted(os.listdir(MODEL_PATH)):
    print("📄", item)

print("\n========== LANGUAGE CONFIRMATION ==========")
print("Tokenizer language: mal")
print("Language: MALAYALAM 🇮🇳")
print("===========================================")

========== CHECKING SAVED MALAYALAM MODEL ==========
Model type: wav2vec2
Vocab size: 136

Model directory contents:
📄 config.json
📄 model.safetensors
📄 preprocessor_config.json
📄 special_tokens_map.json
📄 tokenizer_config.json
📄 training_args.bin
📄 vocab.json

========== LANGUAGE CONFIRMATION ==========
Tokenizer language: mal
Language: MALAYALAM 🇮🇳
